In [1]:
from pydantic import BaseModel, Field, conint, confloat, ConfigDict
from enum import Enum
from typing import Optional, List
from outlines import models, generate
from pydantic import ValidationError
import json
import pandas as pd
from torch.cuda import empty_cache
from ollama import chat
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional
import json
import re

empty_cache()

class SexoJuiz(str, Enum):
    MASCULINO = "Masculino"
    FEMININO = "Feminino"

class SimNao(str, Enum):
    SIM = "Sim"
    NAO = "Não"

class RegimeInicial(str, Enum):
    FECHADO = "Fechado"
    SEMIABERTO = "Semi-aberto"
    ABERTO = "Aberto"
    NONE = "None"

class SentencaModel(BaseModel):
    model_config = ConfigDict(use_enum_values=True)  # Usa valores de Enum em vez de objetos

    processo: str = Field(..., pattern=r'^\d{20}$')
    juiz: str
    sexo_juiz: SexoJuiz
    vara: str
    nome: str
    local: str
    maconha: Optional[str] = "None"
    maconha_g: Optional[confloat(ge=0)] = 0
    cocaina: Optional[str] = "None"
    cocaina_g: Optional[confloat(ge=0)] = 0
    crack: Optional[str] = "None"
    crack_g: Optional[confloat(ge=0)] = 0
    ecstasy: Optional[str] = "None"
    ecstasy_g: Optional[confloat(ge=0)] = 0
    lsd: Optional[str] = "None"
    lsd_g: Optional[confloat(ge=0)] = 0
    outras: Optional[str] = "None"
    anabolizantes: SimNao
    anorexigenos: SimNao
    haxixe: SimNao
    skank: SimNao
    lanca_perfume: SimNao
    tolueno: SimNao
    den_drog: str
    den_outros: Optional[str] = "None"
    sentenca: str
    res_drogas: str
    res_outros: Optional[str] = "None"
    pena_base: str
    agravantes33_agrup: SimNao
    confissao: SimNao
    menoridade: SimNao
    atenuantes33_agrup: SimNao
    adolescente: SimNao
    arma_de_fogo: SimNao
    interestadual: SimNao
    concurso_formal: SimNao
    estabelecimento: SimNao
    aumento33_agrup: SimNao
    paragrafo_4o_agrupado: str
    pena33: str
    pena33_meses: conint(ge=0)
    pena_drogas: str
    pena_outros: Optional[str] = "None"
    tot_pen: str
    tot_pen_meses: conint(ge=0)
    substituicao_da_pena: Optional[str] = "None"
    regime_inicial: RegimeInicial
    flag_local_de_trafico: bool = False
    flag_preso_no_momento_da_sentenca: bool = False
    flag_confissao_informal: bool = False
    flag_confissao: bool = False
    flag_denuncia_anonima: bool = False
    flag_denuncia: bool = False
    flag_atitude_suspeita: bool = False
    flag_divergencias_nos_relatos_dos_policiais: bool = False
    flag_investigacao: bool = False
    flag_interceptacao: bool = False
    flag_mandado: bool = False
    flag_nacionalidade: bool = False
    flag_revista_vexatoria: bool = False
    aval_antecedentes: bool = False
    aval_conduta: bool = False
    aval_personalidade: bool = False
    aval_natureza: bool = False
    aval_quantidade: bool = False
    aval_variedade: bool = False
    aval_circunstancias: bool = False
    aval_consequencias: bool = False
    aval_culpabilidade: bool = False
    

# Modelo Pydantic para múltiplos réus
class SentencaList(BaseModel):
    sentencas: List[SentencaModel]  # Usando a mesma classe SentencaModel definida anteriormente


In [2]:
model_name = 'llama3.2:3b-instruct-fp16' # 'llama3.2:3b-instruct-q8_0'
temperature = 0

In [3]:
schema_json = json.dumps(SentencaList.model_json_schema(), indent=2, ensure_ascii=False)

PROMPT_TEMPLATE = """\
[INST] <<SYS>>
Objetivo:
Você deverá extrair informações de sentenças judiciais de processos criminais brasileiros, com foco em delitos relacionados a tráfico de drogas e correlatos, preenchendo os dados de um dataset estruturado com diversos campos. Cada resposta deve ser produzida para cada par processo/reu – isto é, se na sentença houver mais de um réu, gere um objeto JSON separado para cada par, mantendo o mesmo número de processo para todos, mas com as informações específicas de cada réu.

Contexto:
Você receberá a íntegra de uma sentença judicial (ou de uma ata de audiência contendo a sentença) de um processo criminal no Brasil. O documento pode conter diversas seções: o início com “Vistos” ou “SENTENÇA”, o relatório dos fatos, a fundamentação, a parte dispositiva (decisória) e demais informações, como depoimentos, declarações, referências a denúncias, fundamentos legais e menções à apreensão de drogas. O texto incluirá dados sobre o réu, o juiz, a vara, as penas e diversas flags e avaliações que indicam aspectos processuais específicos.

Instruções de Extração:
1. Extraia os dados para cada par processo/reu. Se na sentença houver mais de um réu, produza um objeto JSON separado para cada par, contendo todas as informações relevantes.
2. Se alguma informação não estiver presente ou não se aplicar, utilize o valor "None" (ou null) para esse campo.
3. Respeite os formatos indicados para cada campo, especialmente para quantidades numéricas, datas e textos jurídicos.
4. Extraia exatamente os campos descritos abaixo, sem omitir ou resumir nenhum detalhe.
5. Nas seções referentes a flags e avaliações (campos com prefixos "flag_" e "aval_"), não se limite à mera correspondência textual exata. Avalie se o texto da sentença descreve as circunstâncias correspondentes aos conceitos abaixo, mesmo que expressos de formas variadas. Utilize seu julgamento para identificar sinônimos, expressões equivalentes ou contextos que indiquem a presença da condição descrita..

**Regras Críticas:**
1. Campos numéricos: Sempre em gramas (apenas números)
2. Campos Sim/Não: Apenas "Sim" ou "Não"
3. Processo: Exatamente 20 dígitos
4. Flags: True apenas se explicitamente mencionado
5. Nada de markdown ou texto extra

**Schema JSON:**
{schema_json}

**Instruções Adicionais:**
- Se houver mais de um réu, gere um objeto JSON separado para cada par processo/reu.
- Se alguma informação não estiver presente, utilize "None" ou null.
- Respeite os formatos indicados para cada campo.
- Avalie o contexto para preencher flags e avaliações, mesmo que expressos de formas variadas.

<</SYS>>

## Documento:
{document}

## Saída (APENAS JSON): 
[/INST]
"""

In [4]:
# Função para extrair JSON da resposta
def extrair_json(resposta: str):
    # Tenta extrair o primeiro bloco JSON da resposta
    json_match = re.search(r'\{.*\}', resposta, re.DOTALL)
    if json_match:
        return json_match.group()
    return None

# Função principal de extração
def extrair_sentencas(documento: str, max_retries=3):
    for _ in range(max_retries):
        try:
            response = chat(
                messages=[{'role': 'user', 'content': PROMPT_TEMPLATE.format(schema_json=schema_json, document=documento)}],
                model= model_name,
                options={'temperature': temperature}  # Reduz a criatividade para maior precisão
            )
            # Extrai o JSON da resposta
            json_str = extrair_json(response.message['content'])
            if json_str:
                return SentencaList.model_validate_json(json_str)
        except (ValidationError, json.JSONDecodeError) as e:
            print(f"Erro de validação (tentativa {_+1}): {e}")
    raise ValueError("Falha após múltiplas tentativas")

In [5]:
# Exemplo de uso
documento = """
Processo nº 00316684320248260051 - Juiz Carlos Almeida (Masculino) - 5ª Vara Criminal
Réus: 
1. João Silva - Apreendidos: 150g de cocaína, 2 armas de fogo. Denúncia base: Art. 33 e 35.
2. Maria Souza - Apreendidos: 50g de maconha. Confissão informal registrada.
Sentença: João - 8 anos regime fechado; Maria - 2 anos regime semiaberto.
"""

In [6]:
df = pd.read_parquet("validation.parquet")[0:2]

documento = df["julgado"].values[0]

print(documento)

SENTENÇA Processo nº: 0031736-90.2017.8.26.0050 - Controle nº 853/17 Classe – Assunto: Procedimento Especial da Lei Antitóxicos - Tráfico de Drogas e Condutas Afins Autor: Justiça Pública Réu: GUSTAVO RODRIGUES PATRICIO VISTOS, etc. GUSTAVO RODRIGUES PATRICIO, qualificado nos autos, foi denunciado como incurso nas sanções do art. 33, caput, da Lei nº 11.343/06, porque, no dia 20 de abril de 2.017, por volta das 02:00 horas, na Rua João da Cunha Lobo, altura do nº 42, Cangaíba, nesta capital, trazia consigo, a consumo de terceiros, 182 pinos de cocaína e 24 invólucros de maconha, substâncias entorpecentes que determinam dependência física ou psíquica, sem autorização e em desacordo com determinação legal e regulamentar. O réu foi notificado para apresentar defesa prévia, a qual foi juntada aos autos por meio de defensora pública. Em seguida, a denúncia foi recebida e o réu citado. Durante a instrução processual foram ouvidas duas testemunhas de acusação e interrogado o réu. Na fase do a

In [7]:
try:
    resultado = extrair_sentencas(documento)
    print(json.dumps(resultado.model_dump(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f"Erro final: {e}")

# Salvando com encoding correto
# with open("sentencas.json", "w", encoding="utf-8") as f:
#     f.write(resultado.model_dump_json(indent=2, ensure_ascii=False))

#print(json.dumps(resultado.model_dump(), indent=2, ensure_ascii=False))

Erro de validação (tentativa 1): 1 validation error for SentencaList
sentencas
  Field required [type=missing, input_value={'decisao': {'improcedenc...es e contradições'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing
Erro de validação (tentativa 2): 1 validation error for SentencaList
sentencas
  Field required [type=missing, input_value={'decisao': {'improcedenc...es e contradições'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing
Erro de validação (tentativa 3): 1 validation error for SentencaList
sentencas
  Field required [type=missing, input_value={'decisao': {'improcedenc...es e contradições'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing
Erro final: Falha após múltiplas tentativas
